# Stage 4: Exploratory Data Analysis (EDA)

Visual analysis of the dataset: intensity distributions, defect area analysis, class statistics, and t-SNE embeddings.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')

DATA_ROOT = Path('../data/mvtec_anomaly_detection')
CATEGORY = 'bottle'
print('EDA notebook ready.')

In [ ]:
# ============================================================
# 4.1 Class Distribution Bar Chart
# ============================================================
def plot_class_distribution(data_root, category):
    test_path = data_root / category / 'test'
    counts = {}
    for d in sorted(test_path.iterdir()):
        if d.is_dir():
            counts[d.name] = len(list(d.glob('*.png')))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Per defect type
    colors = ['#2ecc71' if k == 'good' else '#e74c3c' for k in counts.keys()]
    ax1.bar(counts.keys(), counts.values(), color=colors, edgecolor='black', linewidth=0.5)
    ax1.set_title(f'Image Count per Defect Type\n({category})', fontsize=14)
    ax1.set_xlabel('Defect Type')
    ax1.set_ylabel('Count')
    ax1.tick_params(axis='x', rotation=45)
    
    normal_patch = mpatches.Patch(color='#2ecc71', label='Normal')
    defect_patch = mpatches.Patch(color='#e74c3c', label='Defective')
    ax1.legend(handles=[normal_patch, defect_patch])
    
    # Binary class pie
    n_normal = counts.get('good', 0)
    n_defect = sum(v for k, v in counts.items() if k != 'good')
    ax2.pie([n_normal, n_defect], labels=['Normal', 'Defective'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'black'})
    ax2.set_title('Normal vs Defective (Test Set)', fontsize=14)
    
    plt.tight_layout()
    plt.savefig('../assets/eda_class_distribution.png', dpi=150)
    plt.show()

if DATA_ROOT.exists():
    plot_class_distribution(DATA_ROOT, CATEGORY)
else:
    print('Dataset not found — run with downloaded MVTec AD')

In [ ]:
# ============================================================
# 4.2 Pixel Intensity Distribution (Normal vs Defective)
# ============================================================
def plot_intensity_distributions(data_root, category, n=50):
    normal_path  = data_root / category / 'train' / 'good'
    defect_paths = [p for p in (data_root / category / 'test').iterdir()
                    if p.is_dir() and p.name != 'good']
    
    normal_pixels, defect_pixels = [], []
    
    for img_path in list(normal_path.glob('*.png'))[:n]:
        img = np.array(Image.open(img_path).convert('L').resize((224, 224)))
        normal_pixels.extend(img.flatten())
    
    for dp in defect_paths:
        for img_path in list(dp.glob('*.png'))[:n//len(defect_paths)+1]:
            img = np.array(Image.open(img_path).convert('L').resize((224, 224)))
            defect_pixels.extend(img.flatten())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram overlay
    axes[0].hist(normal_pixels, bins=64, alpha=0.6, color='#2ecc71', label='Normal', density=True)
    axes[0].hist(defect_pixels, bins=64, alpha=0.6, color='#e74c3c', label='Defective', density=True)
    axes[0].set_title('Pixel Intensity Distribution', fontsize=13)
    axes[0].set_xlabel('Pixel Value (0-255)')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    
    # Box plot
    data = [normal_pixels[:5000], defect_pixels[:5000]]
    bp = axes[1].boxplot(data, labels=['Normal', 'Defective'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')
    axes[1].set_title('Pixel Value Distribution (Box Plot)', fontsize=13)
    axes[1].set_ylabel('Pixel Value')
    
    plt.tight_layout()
    plt.savefig('../assets/eda_intensity.png', dpi=150)
    plt.show()
    
    print(f'Normal  mean: {np.mean(normal_pixels):.1f}, std: {np.std(normal_pixels):.1f}')
    print(f'Defect  mean: {np.mean(defect_pixels):.1f}, std: {np.std(defect_pixels):.1f}')

if DATA_ROOT.exists():
    plot_intensity_distributions(DATA_ROOT, CATEGORY)

In [ ]:
# ============================================================
# 4.3 Defect Region Area Analysis using Ground Truth Masks
# ============================================================
def analyze_defect_areas(data_root, category):
    mask_root = data_root / category / 'ground_truth'
    if not mask_root.exists():
        print('Ground truth masks not found.')
        return
    
    areas, defect_types = [], []
    for dtype_dir in sorted(mask_root.iterdir()):
        if not dtype_dir.is_dir():
            continue
        for mask_path in dtype_dir.glob('*_mask.png'):
            mask = np.array(Image.open(mask_path).convert('L'))
            defect_area = (mask > 128).sum() / mask.size * 100  # % of image
            areas.append(defect_area)
            defect_types.append(dtype_dir.name)
    
    df = pd.DataFrame({'defect_type': defect_types, 'defect_area_pct': areas})
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Box plot per defect type
    df.boxplot(column='defect_area_pct', by='defect_type', ax=axes[0])
    axes[0].set_title('Defect Region Area by Type')
    axes[0].set_xlabel('Defect Type')
    axes[0].set_ylabel('Defect Area (% of image)')
    plt.sca(axes[0])
    plt.xticks(rotation=45)
    
    # Histogram
    axes[1].hist(areas, bins=20, color='#e74c3c', edgecolor='black', alpha=0.8)
    axes[1].axvline(np.mean(areas), color='navy', linestyle='--', label=f'Mean: {np.mean(areas):.1f}%')
    axes[1].set_title('Distribution of Defect Sizes')
    axes[1].set_xlabel('Defect Area (% of image)')
    axes[1].set_ylabel('Count')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('../assets/eda_defect_areas.png', dpi=150)
    plt.show()
    
    print(df.groupby('defect_type')['defect_area_pct'].describe().round(2))

if DATA_ROOT.exists():
    analyze_defect_areas(DATA_ROOT, CATEGORY)